# Multi-Class Jet Tagging

In this tutorial, we move from a simple binary classification problem to a more realistic multi-class physics task: identifying jets produced at the LHC with a neural network.

Just like in the first tutorial, we will train, evaluate, and compare models step by step, but this time the problem is more challenging. Instead of predicting only two classes, our model must distinguish between five jet types using high-level physics features.

We use the hls4ml LHC jet dataset from OpenML. \
It contains about 830,000 jet events, 16 high-level input features, and 5 jet classes:
- quark (q),
- gluon (g),
- W boson (w),
- Z boson (z),
- top quark (t).

In this tutorial, we will focus on:

- Multi-class neural network training and evaluation

- Class prediction probabilities and confusion between jet classes

- ROC/AUC-style performance comparisons for multi-class classification


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nairods/Scies4Free-FastML-tutorial/blob/main/Multi-Class_Classification_NN.ipynb)

# 1. Import Libraries

In [1]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.utils import to_categorical
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

%matplotlib inline
RANDOM_STATE = 42

## 2. Load and Inspect the Dataset


In [2]:
data = fetch_openml('hls4ml_lhc_jets_hlf')
X, y = data['data'], data['target']

print("Feature names:")
print(list(data["feature_names"]))

print("\nInput shape and target shape:")
print("X:", X.shape)
print("y:", y.shape)

print("\nFirst 5 input rows:")
display(X.head())

print("\nFirst 5 labels:")
display(y.head())

print("\nClass names:")
print(y.cat.categories.tolist())

Feature names:
['zlogz', 'c1_b0_mmdt', 'c1_b1_mmdt', 'c1_b2_mmdt', 'c2_b1_mmdt', 'c2_b2_mmdt', 'd2_b1_mmdt', 'd2_b2_mmdt', 'd2_a1_b1_mmdt', 'd2_a1_b2_mmdt', 'm2_b1_mmdt', 'm2_b2_mmdt', 'n2_b1_mmdt', 'n2_b2_mmdt', 'mass_mmdt', 'multiplicity']

Input shape and target shape:
X: (830000, 16)
y: (830000,)

First 5 input rows:


,zlogz,c1_b0_mmdt,c1_b1_mmdt,c1_b2_mmdt,c2_b1_mmdt,c2_b2_mmdt,d2_b1_mmdt,d2_b2_mmdt,d2_a1_b1_mmdt,d2_a1_b2_mmdt,m2_b1_mmdt,m2_b2_mmdt,n2_b1_mmdt,n2_b2_mmdt,mass_mmdt,multiplicity
0,-2.935125,0.383155,0.005126,0.000084,0.009070,0.000179,1.769445,2.123898,1.769445,0.308185,0.135687,0.083278,0.412136,0.299058,8.926882,75.0
1,-1.927335,0.270699,0.001585,0.000011,0.003232,0.000029,2.038834,2.563099,2.038834,0.211886,0.063729,0.036310,0.310217,0.226661,3.886512,31.0
2,-3.112147,0.458171,0.097914,0.028588,0.124278,0.038487,1.269254,1.346238,1.269254,0.246488,0.115636,0.079094,0.357559,0.289220,162.144669,61.0
3,-2.666515,0.437068,0.049122,0.007978,0.047477,0.004802,0.966505,0.601864,0.966505,0.160756,0.082196,0.033311,0.238871,0.094516,91.258934,39.0
4,-2.484843,0.428981,0.041786,0.006110,0.023066,0.001123,0.552002,0.183821,0.552002,0.084338,0.048006,0.014450,0.141906,0.036665,79.725777,35.0



First 5 labels:


,class
0,g
1,w
2,t
3,z
4,w



Class names:
['g', 'q', 't', 'w', 'z']


## Feature explanation

| Feature       | Meaning                                                                                                                       |
| ------------- | ----------------------------------------------------------------------------------------------------------------------------- |
| zlogz         | A jet substructure variable related to how momentum is distributed among particles inside the jet.              |
| c1_b0_mmdt    | Energy correlation feature C1 with parameter setting β=0,  mainly sensitive to the overall radiation pattern of the jet.          |
| c1_b1_mmdt    | Energy correlation feature C1 with β=1. |
| c1_b2_mmdt    | Energy correlation feature C1 with β=2. |
| c2_b1_mmdt    | Energy correlation feature C2 with β=1, used to describe jet shape and prong structure.          |
| c2_b2_mmdt    | Energy correlation feature C2 with β=2.        |
| d2_b1_mmdt    | Energy correlation feature D2 with β=1, used to distinguish one-prong jets from two-prong or more complex jets.|
| d2_b2_mmdt    | Energy correlation feature D2 with β=2.  |
| d2_a1_b1_mmdt | Variant of the D2 observable with additional parameter choices. |
| d2_a1_b2_mmdt | Another variant of the D2 observable with different parameter settings.                 |
| m2_b1_mmdt    | Energy correlation feature M2 with β=1, used to describe finer details of jet substructure.   |
| m2_b2_mmdt    | Energy correlation feature M2 with β=2. |
| n2_b1_mmdt    | Energy correlation feature N2 with β=1, sensitive to the internal prong structure of a jet.      |
| n2_b2_mmdt    | Energy correlation feature N2 with β=2.                 |
| mass_mmdt     | Groomed jet mass after the modified mass-drop tagger (mMDT) procedure.                 |
| multiplicity  | Number of constituents or particles reconstructed inside the jet.       |
| target        | Jet class label: g = gluon, q = quark, t = top quark, w = W boson, z = Z boson.                       |

\
β controls how strongly the observable depends on the angular separation of particles inside the jet:

smaller β is less angle-sensitive, while larger β emphasizes the jet’s geometric structure more strongly.

## 3. Encode Labels and Split Dataset

As you saw above, the `y` target is an array of strings, e.g. \['g', 'w',...\] etc.  
We map them to integers and split the dataset into training and validation sets:

- 70% training
- 15% validation
- 15% test

with equal representation of classes by using stratify.

In [3]:
# @title
import ipywidgets as widgets
from IPython.display import display, Markdown
import re

display(Markdown("### Question 1"))
display(Markdown("What is the purpose of the various datasets?"))

answer = widgets.Textarea(
    placeholder="Write your answer here...",
    layout=widgets.Layout(width="700px", height="140px")
)

button = widgets.Button(description="Check answer", button_style="info")
output = widgets.Output()

def contains_any(text, keywords):
    return any(k in text for k in keywords)

def check_answer(b):
    user = answer.value.strip().lower()
    user = re.sub(r"\s+", " ", user)

    train_ok = (
        contains_any(user, ["train", "training"]) and
        contains_any(user, ["fit", "fitting", "train the model", "learn", "training the model", "trains the model", "teach the model"]) and
        contains_any(user, ["weights", "parameters", "model parameters", "train the model", "training the model", "trains the model"])
    )

    val_ok = (
        contains_any(user, ["val", "validation"]) and
        contains_any(user, ["hyperparameter", "hyperparameters", "performance", "model parameters", "the model"]) and
        contains_any(user, ["learn", "tune", "tuning", "improve", "improving", "monitor", "adjust", "adjusting"]) and
        contains_any(user, ["performance", "overfit", "development", "without influencing the training", "early-stopping", "early stopping", "select the model", "choose the model", "best model", "model selection", "overfitting", "tune the model"])
    )

    test_ok = (
        contains_any(user, ["test", "testing"]) and
        contains_any(user, ["final", "finalized", "at the end", "last", "never seen", "new data", "unseen", "held-out", "not used in training", "only", "solely", "unbiased", "separate"]) and
        contains_any(user, ["measure", "measures", "reporting", "report", "testing", "compare", "comparing", "comparison", "check", "checking", "test", "evaluation", "evaluating", "evaluate"])
    )

    with output:
        output.clear_output()
        if train_ok and val_ok and test_ok:
            print("Correct!")
            print("Train: used to fit model parameters (weights/trees).")
            print("Validation: used to tune hyperparameters and for early stopping / model selection.")
            print("Test: used once at the end for final reporting and comparison.")
        else:
            print("Not quite yet.")
            print("Make sure you describe each dataset separately and explain its role clearly.")

button.on_click(check_answer)

display(answer, button, output)


### Question 1

What is the purpose of the various datasets?

Textarea(value='', layout=Layout(height='140px', width='700px'), placeholder='Write your answer here...')

Button(button_style='info', description='Check answer', style=ButtonStyle())

Output()

In [4]:
# Convert y strings into integers
codecs = {'g': 0, 'q': 1, 't': 4, 'w': 2, 'z': 3}
y = np.array([codecs[i] for i in y])

# Split data in train vs val_test
X_train, X_val_test, y_train, y_val_test = train_test_split(X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y)

# Split val_test in half
X_val, X_test, y_val, y_test = train_test_split(X_val_test, y_val_test, test_size=0.5, random_state=RANDOM_STATE, stratify=y_val_test)

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

Train shape: (581000, 16)
Validation shape: (124500, 16)
Test shape: (124500, 16)


## 4. Feature Scaling

Neural networks train better when input features are scaled / standardized.

We:
- Fit the scaler on training data only
- Apply the same transformation to validation and test sets

In [5]:
# Convert to float32
X_train = X_train.astype(np.float32)
X_val   = X_val.astype(np.float32)
X_test  = X_test.astype(np.float32)

# Scale using training data only
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

In [6]:
# @title
import ipywidgets as widgets
from IPython.display import display, Markdown
import re

display(Markdown("### Question 2"))
display(Markdown("Why do neural networks train better when input features are standardized?"))

answer = widgets.Textarea(
    placeholder="Write your answer here...",
    layout=widgets.Layout(width="700px", height="140px")
)

button = widgets.Button(description="Check answer", button_style="info")
output = widgets.Output()

def contains_any(text, keywords):
    return any(k in text for k in keywords)

def check_answer(b):
    user = answer.value.strip().lower()
    user = re.sub(r"\s+", " ", user)

    scale_ok = contains_any(user, [
        "same scale", "comparable scale", "similar scale", "same scales", "comparable scales", "similar scales","standardized",
        "zero mean", "unit variance", "normalized", "center around zero", "centers around zero", "similar range", "similar ranges", "symmetric"
    ])

    train_ok = contains_any(user, [
        "train faster", "faster training", "more stable", "stable training", "stabilize",
        "converge better", "better convergence", "optimization", "optimizer", "stabilizes",
        "gradient descent", "gradients", "converge faster", "more reliably", "more reliable", "gradient-based",
        "balanced", "more efficient", "more efficiently"
    ])

    dominate_ok = contains_any(user, [
        "no feature dominates", "prevent one feature from dominating",
        "large-magnitude feature dominates", "large feature dominates", "large value features dominate", "large value feature dominates",
        "balanced gradients", "features contribute more equally",
        "one input can dominate", "dominate the gradients", "equal contribution", "prevents features with large values from dominating", "prevent features with large values from dominating"
    ])

    with output:
        output.clear_output()
        if scale_ok and train_ok and dominate_ok:
            print("Correct!")
            print("Standardization helps neural networks train faster and more stably")
            print("because gradient-based optimizers work better when input features")
            print("are on a comparable scale, so no large-magnitude feature dominates the gradients.")
        else:
            print("Not quite yet.")
            if not scale_ok:
                print("- Hint: mention comparable feature scales")

            if not train_ok:
                print("- Hint: explain how standardization helps gradient-based training")

            if not dominate_ok:
                print("- Hint: mention why large-magnitude features can cause problems")

button.on_click(check_answer)

display(answer, button, output)

### Question 2

Why do neural networks train better when input features are standardized?

Textarea(value='', layout=Layout(height='140px', width='700px'), placeholder='Write your answer here...')

Button(button_style='info', description='Check answer', style=ButtonStyle())

Output()

In [7]:
# @title
import ipywidgets as widgets
from IPython.display import display, Markdown
import re

display(Markdown("### Question 3"))
display(Markdown("Why do we fit the scaler on training data only?"))

answer = widgets.Textarea(
    placeholder="Write your answer here...",
    layout=widgets.Layout(width="700px", height="140px")
)

button = widgets.Button(description="Check answer", button_style="info")
output = widgets.Output()

def contains_any(text, keywords):
    return any(k in text for k in keywords)

def check_answer(b):
    user = answer.value.strip().lower()
    user = re.sub(r"\s+", " ", user)

    prevent_ok = contains_any(user, [
        "prevent", "avoid", "reduce", "limit", "minimize", "stop", "block", "protect against", "keep from", "only the training data", "only from the data the model is allowed to", "only data the model should"
    ])

    leak_ok = contains_any(user, [
        "leakage", "leak", "mixing", "leaking", "adding", "combining", "mix", "combine", "add", "computed only", "gain", "influence", "biases", "biased", "bias"
    ])

    data_ok = (
        contains_any(user, [
            "data", "mean and std", "mean/std", "datasets", "dataset",
            "statistics", "info", "information", "parameters", "mean and standard deviation"
        ])
    )

    valtest_ok = contains_any(user, [
        "validation", "test", "val/test", "validation/test", "unseen data", "other data", "val or test"
    ])

    fairness_ok = contains_any(user, [
        "fair", "honest", "unbiased", "good", "reliable", "trustworthy", "best", "correctly",
        "realistic", "proper", "final", "preprocessing", "true", "real", "correct", "unbiased"
    ])

    evaluation_ok = contains_any(user, [
        "evaluated", "process", "evaluation", "preprocessing", "check", "test", "checking", "testing", "evaluating", "evaluate", "preprocess", "pre-process", "performance"
    ])

    with output:
        output.clear_output()
        if prevent_ok and leak_ok and data_ok and valtest_ok and fairness_ok and evaluation_ok:
            print("Correct!")
            print("We fit the scaler on the training data only to avoid data leakage.")
            print("If we learn the mean/std from validation or test data,")
            print("information from those sets leaks into preprocessing,")
            print("which makes the final evaluation less fair and less realistic.")
        else:
            print("Not quite yet.")

            if not prevent_ok or not leak_ok or not data_ok:
                print("- Hint: mention why we should prevent data leakage.")

            if not valtest_ok:
                print("- Hint: mention validation and/or test data explicitly.")

            if not fairness_ok or not evaluation_ok:
                print("- Hint: explain why this matters for a fair or realistic final evaluation.")

button.on_click(check_answer)

display(answer, button, output)

### Question 3

Why do we fit the scaler on training data only?

Textarea(value='', layout=Layout(height='140px', width='700px'), placeholder='Write your answer here...')

Button(button_style='info', description='Check answer', style=ButtonStyle())

Output()

In case the automatic download does not work, a local backup of the dataset that can be loaded instead:

In [11]:
!git clone https://github.com/nairods/Scies4Free-FastML-tutorial.git
# DATA_PATH = "Scies4Free-FastML-tutorial/data/jet_tagging/"

# train = np.load(DATA_PATH + "train.npz", allow_pickle=True)
# val   = np.load(DATA_PATH + "val.npz", allow_pickle=True)
# test  = np.load(DATA_PATH + "test.npz", allow_pickle=True)

# feature_names = ['zlogz', 'c1_b0_mmdt', 'c1_b1_mmdt', 'c1_b2_mmdt', 'c2_b1_mmdt', 'c2_b2_mmdt', 'd2_b1_mmdt', 'd2_b2_mmdt', 'd2_a1_b1_mmdt', 'd2_a1_b2_mmdt', 'm2_b1_mmdt', 'm2_b2_mmdt', 'n2_b1_mmdt', 'n2_b2_mmdt', 'mass_mmdt', 'multiplicity']
# print(feature_names)

# X_train, y_train = train["X"], train["y"]
# X_val,   y_val   = val["X"],   val["y"]
# X_test,  y_test  = test["X"],  test["y"]

Cloning into 'Scies4Free-FastML-tutorial'...
remote: Enumerating objects: 249, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 249 (delta 8), reused 1 (delta 1), pack-reused 236 (from 1)
Receiving objects: 100% (249/249), 94.70 MiB | 19.42 MiB/s, done.
Resolving deltas: 100% (112/112), done.


We need to convert the class labels into one-hot vectors so that they match the softmax output of the network  
and can be properly compared using categorical cross-entropy loss.

One-hot encoding represents each class as a vector with a 1 at the correct class position and 0 elsewhere:
- class 0 -> (1,0,0,0,0)
- class 1 -> (0,1,0,0,0)
- class 2 -> (0,0,1,0,0)
- class 3 -> (0,0,0,1,0)
- class 4 -> (0,0,0,0,1)

In [9]:
# Convert Labels to One-Hot representation
y_train = to_categorical(y_train, 5)
y_val = to_categorical(y_val, 5)
y_test = to_categorical(y_test, 5)

# 2. Build the Neural Network

Architecture:
- 3 hidden layers with 64, then 32, then 32 neurons
- Each with a ReLU activation
- 5 output neurons (one for each class)
- Finish with a Softmax activation

In [15]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l1
import sys
repo_path = "/content/Scies4Free-FastML-tutorial"
sys.path.insert(0, repo_path)
from callbacks import all_callbacks

In [ ]:
model = Sequential()
model.add(Dense(64, input_shape=(16,), name='fc1', kernel_initializer='lecun_uniform', kernel_regularizer=l1(0.0001)))
model.add(Activation(activation='relu', name='relu1'))
model.add(Dense(32, name='fc2', kernel_initializer='lecun_uniform', kernel_regularizer=l1(0.0001)))
model.add(Activation(activation='relu', name='relu2'))
model.add(Dense(32, name='fc3', kernel_initializer='lecun_uniform', kernel_regularizer=l1(0.0001)))
model.add(Activation(activation='relu', name='relu3'))
model.add(Dense(5, name='output', kernel_initializer='lecun_uniform', kernel_regularizer=l1(0.0001)))
model.add(Activation(activation='softmax', name='softmax'))

## 2.1 Train the model
We train the neural network using the Adam optimizer and categorical cross-entropy loss.

During training:
- The learning rate is automatically reduced when the validation performance plateaus.
- The best-performing model is saved to the directory `model_1`.

The network architecture is relatively small, so training should complete within a few minutes, even on a CPU.

If you have already trained the model and restarted the notebook kernel, you can set `train = False` to skip training and load the previously saved model instead.

In [ ]:
train = True
if train:
    adam = Adam(lr=0.0001)
    model.compile(optimizer=adam, loss=['categorical_crossentropy'], metrics=['accuracy'])
    callbacks = all_callbacks(
        stop_patience=1000,
        lr_factor=0.5,
        lr_patience=10,
        lr_epsilon=0.000001,
        lr_cooldown=2,
        lr_minimum=0.0000001,
        outputDir='model_1',
    )

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=1024,
        epochs=10,
        shuffle=True
    )
else:
    from tensorflow.keras.models import load_model

    model = load_model('model_1/KERAS_check_best_model.h5')

## 2.2 Check performance
Check the accuracy and make a Receiver Operating Characteristic (ROC) curve

In [ ]:
import plotting
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

y_keras = model.predict(X_test)
print("Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_keras, axis=1))))
plt.figure(figsize=(9, 9))
_ = plotting.makeRoc(y_test, y_keras, le.classes_)

# 3. Convert to FPGA Firmware with hls4ml

Next, we will convert the trained Keras model into a **low-latency FPGA implementation** using `hls4ml`.  
First, we will check that the classification performance remains accurate when using **fixed-point data types**.  
Then, we will synthesize the model with **Vitis HLS** and inspect its **latency and FPGA resource usage**.

### Create an hls4ml configuration and model

The `hls4ml` library uses a **configuration dictionary** to control how the neural network is translated to FPGA firmware.  
Here, we will show a **simple configuration**, however more advanced settings are possible.

In [ ]:
import hls4ml

config = hls4ml.utils.config_from_keras_model(model, granularity='model', backend='Vitis')
print("-----------------------------------")
print("Configuration")
plotting.print_dict(config)
print("-----------------------------------")
hls_model = hls4ml.converters.convert_from_keras_model(
    model, hls_config=config, backend='Vitis', output_dir='model_1/hls4ml_prj', part='xcu250-figd2104-2L-e'
)

Let's visualise what we created. The model architecture is shown, annotated with the shape and data types

In [ ]:
hls4ml.utils.plot_model(hls_model, show_shapes=True, show_precision=True, to_file=None)

## 3.1 Compile, predict
Now we need to check that this model performance is still good. We compile the hls_model, and then use `hls_model.predict` to execute the FPGA firmware with bit-accurate emulation on the CPU.

In [ ]:
hls_model.compile()
X_test = np.ascontiguousarray(X_test)
y_hls = hls_model.predict(X_test)

## 3.2 Compare
That was easy! Now let's see how the performance compares to Keras:

In [ ]:
print("Keras  Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_keras, axis=1))))
print("hls4ml Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_hls, axis=1))))

fig, ax = plt.subplots(figsize=(9, 9))
_ = plotting.makeRoc(y_test, y_keras, le.classes_)
plt.gca().set_prop_cycle(None)  # reset the colors
_ = plotting.makeRoc(y_test, y_hls, le.classes_, linestyle='--')

from matplotlib.lines import Line2D

lines = [Line2D([0], [0], ls='-'), Line2D([0], [0], ls='--')]
from matplotlib.legend import Legend

leg = Legend(ax, lines, labels=['keras', 'hls4ml'], loc='lower right', frameon=False)
ax.add_artist(leg)

# 4. Synthesize
Now we will use **Vitis HLS** to synthesize the model into FPGA firmware.  
We can start the build directly from our `hls_model` object.  

After synthesis, the generated IP can be integrated into a workflow to compile for a specific FPGA board.  
For this tutorial, we will focus on **reviewing the reports** generated by Vitis HLS, paying attention to **latency** and **resource usage**.

**Note:** This step can take several minutes.

In [ ]:
hls_model.build(csim=False)

## 4.1 Check the reports
Print out the reports generated by Vitis HLS. Pay attention to the Latency and the 'Utilization Estimates' sections

In [ ]:
hls4ml.report.read_vivado_report('model_1/hls4ml_prj/')